### Performance testing

- Dask inevitably comes with an overhead (starting and scheduling tasks)
- At some point, the gain from parallelism is higher than the overhead
- But when?

In [1]:
import pandas as pd
import dask.dataframe as dd
from statistics import mean

# A record every six seconds
nb_files = 600

from dask.distributed import Client

# getting CSV filenames
filenames = ['boats'+str(i)+'.csv' for i in range(nb_files)]

# starting the cluster
client = Client(n_workers=12)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 12
Total threads: 24,Total memory: 30.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44277,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40433,Total threads: 2
Dashboard: http://127.0.0.1:36447/status,Memory: 2.57 GiB
Nanny: tcp://127.0.0.1:46245,


In [2]:
import time

# firstly with pandas


# loading data (Pandas style)
df_list = []
loading_start_time = time.time()

for fn in filenames:
    df = pd.read_csv(fn)
    df_list.append(df)

global_df = pd.concat(df_list, ignore_index=True)
loading_end_time = time.time()

global_df
print ("Loading time: " + str((loading_end_time-loading_start_time)*1000) + " ms.")

Loading time: 44556.96153640747 ms.


In [ ]:
# computing the avg speed 25 times
runtimes = []

for i in range(25):
    start_time = time.time()
    global_speed_avgs = global_df["Speed"].mean()
    print ("Mean speed of boat: " + str(global_speed_avgs))
    end_time = time.time()
    runtimes.append(end_time-start_time)

print ("Avg compute time: " + str((sum(runtimes) / len(runtimes)) * 1000) + " ms.")

Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Mean speed of boat: 26.410568783333332
Avg compute time: 142.033

### speed up
- 20000 boats * 600 time steps (12 000 000 rows). Pandas = 4.5s, Dask = 2s
- 10000 boats * 600 time steps (6 000 000 rows). Pandas = 2.5s, Dask < 2s
- 5000 boats * 600 time steps (3 000 000 rows). Pandas = 1.7s, Dask = 1.7s

In [4]:
%%time

# then with dask DataFrame

loading_start_time = time.time()
ddf = dd.read_csv('boats*.csv')
loading_end_time = time.time()

dask_runtimes = []
for i in range(5):
    dask_run_start_time = time.time()
    mean_speed = ddf["Speed"].mean()
    ms = mean_speed.compute()
    dask_run_end_time = time.time()
    dask_runtimes.append(dask_run_end_time - dask_run_start_time)

print ("Mean speed of boat: " + str(ms))
print ("Dask loading time: " + str ((loading_end_time - loading_start_time) * 1000) + "ms.")
print ("Avg compute time: " + str((sum(dask_runtimes) / len(dask_runtimes)) * 1000) + " ms.")

Mean speed of boat: 26.41053731577114
Dask loading time: 197.1876621246338ms.
Avg compute time: 6998.283338546753 ms.
CPU times: user 26.7 s, sys: 2.5 s, total: 29.2 s
Wall time: 35.2 s


In [45]:
client.shutdown()
client.close()